In [9]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from typing import Tuple, Dict, List, Optional
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import warnings


In [10]:
# Random seed for reproducibility
np.random.seed(42)
n_cases = 1000

# Define medical conditions with MORE REALISTIC symptom patterns
conditions = {
    'migraine': {
        'symptoms': [
            ('severe_headache', (7, 10), (4, 72), ['one_side', 'temples', 'behind_eyes', 'whole_head']),
            ('throbbing_pain', (7, 10), (4, 72), ['head']),
            ('nausea', (5, 9), (2, 48), [None]),
            ('vomiting', (6, 9), (1, 24), [None]),
            ('light_sensitivity', (6, 10), (4, 72), ['eyes']),
            ('sound_sensitivity', (5, 9), (4, 72), ['ears']),
            ('visual_aura', (4, 8), (0.5, 2), ['vision']),
            ('dizziness', (4, 8), (2, 24), ['head']),
            ('neck_stiffness', (4, 7), (4, 48), ['neck']),
            ('scalp_tenderness', (3, 7), (4, 72), ['scalp']),
            ('blurred_vision', (4, 7), (1, 24), ['eyes']),
            ('fatigue', (5, 8), (4, 72), ['whole_body']),
            ('difficulty_concentrating', (5, 8), (4, 48), [None]),
            ('irritability', (4, 7), (4, 72), [None])
        ],
        'symptom_count': (3, 8),
        'age_range': (15, 65),
        'confidence_range': (0.75, 0.95)
    },
    'influenza': {
        'symptoms': [
            ('high_fever', (7, 10), (48, 168), ['whole_body']),
            ('body_aches', (7, 10), (48, 168), ['muscles', 'joints', 'back']),
            ('severe_fatigue', (8, 10), (72, 336), ['whole_body']),
            ('dry_cough', (6, 9), (72, 240), ['chest', 'throat']),
            ('sore_throat', (5, 8), (48, 120), ['throat']),
            ('chills', (6, 9), (24, 96), ['whole_body']),
            ('sweating', (5, 8), (24, 120), ['whole_body']),
            ('headache', (5, 8), (48, 120), ['whole_head', 'forehead']),
            ('nasal_congestion', (4, 7), (72, 168), ['nose', 'sinuses']),
            ('runny_nose', (3, 7), (48, 168), ['nose']),
            ('chest_discomfort', (5, 8), (48, 168), ['chest']),
            ('weakness', (7, 10), (72, 240), ['whole_body']),
            ('loss_appetite', (5, 8), (48, 168), [None]),
            ('mild_nausea', (3, 6), (24, 96), [None]),
            ('sneezing', (3, 6), (48, 120), ['nose'])
        ],
        'symptom_count': (5, 10),
        'age_range': (5, 80),
        'confidence_range': (0.80, 0.95)
    },
    'tension_headache': {
        'symptoms': [
            ('dull_headache', (3, 7), (1, 48), ['whole_head', 'forehead', 'back_of_head']),
            ('pressure_sensation', (4, 7), (2, 48), ['temples', 'forehead', 'band_around_head']),
            ('neck_pain', (3, 7), (2, 72), ['neck', 'upper_neck']),
            ('shoulder_tension', (3, 7), (2, 48), ['shoulders', 'upper_back']),
            ('scalp_tenderness', (2, 5), (2, 48), ['scalp']),
            ('mild_fatigue', (3, 6), (4, 48), ['whole_body']),
            ('difficulty_sleeping', (3, 6), (12, 72), [None]),
            ('jaw_tightness', (2, 5), (2, 24), ['jaw']),
            ('eye_strain', (3, 6), (2, 24), ['eyes']),
            ('irritability', (3, 6), (4, 48), [None])
        ],
        'symptom_count': (3, 6),
        'age_range': (18, 70),
        'confidence_range': (0.65, 0.85)
    },
    'food_poisoning': {
        'symptoms': [
            ('severe_nausea', (7, 10), (2, 48), ['stomach']),
            ('vomiting', (7, 10), (2, 36), ['stomach']),
            ('watery_diarrhea', (7, 10), (6, 48), ['intestines']),
            ('abdominal_cramps', (6, 9), (6, 48), ['stomach', 'lower_abdomen', 'intestines']),
            ('stomach_pain', (6, 9), (4, 48), ['stomach', 'upper_abdomen']),
            ('fever', (4, 8), (12, 48), ['whole_body']),
            ('chills', (4, 7), (12, 48), ['whole_body']),
            ('weakness', (6, 9), (12, 72), ['whole_body']),
            ('loss_appetite', (6, 9), (12, 72), [None]),
            ('dehydration', (5, 8), (12, 48), ['whole_body']),
            ('headache', (4, 7), (6, 48), ['whole_head']),
            ('dizziness', (4, 7), (12, 48), ['head']),
            ('bloating', (4, 7), (6, 48), ['abdomen']),
            ('gas', (3, 6), (6, 48), ['intestines'])
        ],
        'symptom_count': (4, 8),
        'age_range': (10, 80),
        'confidence_range': (0.70, 0.90)
    },
    'anxiety_attack': {
        'symptoms': [
            ('chest_tightness', (7, 10), (0.25, 2), ['chest']),
            ('racing_heart', (8, 10), (0.25, 2), ['chest']),
            ('shortness_breath', (7, 10), (0.25, 2), ['chest', 'lungs']),
            ('hyperventilation', (6, 9), (0.25, 1), ['lungs']),
            ('sweating', (6, 9), (0.25, 2), ['whole_body', 'palms', 'forehead']),
            ('trembling', (6, 9), (0.25, 2), ['hands', 'legs', 'whole_body']),
            ('dizziness', (6, 9), (0.25, 2), ['head']),
            ('feeling_faint', (5, 9), (0.25, 1), [None]),
            ('nausea', (5, 8), (0.25, 2), ['stomach']),
            ('chills', (5, 8), (0.25, 2), ['whole_body']),
            ('hot_flashes', (5, 8), (0.25, 1), ['whole_body']),
            ('tingling', (4, 7), (0.25, 2), ['hands', 'feet', 'face']),
            ('sense_of_dread', (8, 10), (0.25, 2), [None]),
            ('feeling_detached', (6, 9), (0.25, 2), [None]),
            ('fear_of_dying', (7, 10), (0.25, 2), [None])
        ],
        'symptom_count': (5, 10),
        'age_range': (15, 60),
        'confidence_range': (0.60, 0.85)
    },
    'strep_throat': {
        'symptoms': [
            ('severe_sore_throat', (8, 10), (24, 120), ['throat']),
            ('painful_swallowing', (7, 10), (24, 96), ['throat']),
            ('fever', (6, 9), (24, 120), ['whole_body']),
            ('swollen_lymph_nodes', (6, 9), (24, 120), ['neck', 'under_jaw']),
            ('red_swollen_tonsils', (6, 9), (24, 96), ['tonsils']),
            ('white_patches_throat', (5, 8), (24, 96), ['throat', 'tonsils']),
            ('tiny_red_spots_palate', (4, 7), (24, 72), ['roof_of_mouth']),
            ('headache', (4, 7), (12, 72), ['whole_head']),
            ('body_aches', (4, 7), (24, 96), ['muscles']),
            ('nausea', (3, 6), (12, 48), [None]),
            ('vomiting', (3, 6), (12, 48), [None]),
            ('loss_appetite', (4, 7), (24, 96), [None]),
            ('fatigue', (5, 8), (24, 120), ['whole_body'])
        ],
        'symptom_count': (4, 8),
        'age_range': (5, 40),
        'confidence_range': (0.75, 0.92)
    },
    'uti': {
        'symptoms': [
            ('burning_urination', (7, 10), (12, 96), ['urethra', 'bladder']),
            ('frequent_urination', (7, 9), (12, 96), ['bladder']),
            ('urgent_urination', (6, 9), (12, 96), ['bladder']),
            ('lower_abdominal_pain', (5, 8), (12, 96), ['lower_abdomen']),
            ('pelvic_pressure', (5, 8), (12, 96), ['pelvis']),
            ('cloudy_urine', (4, 7), (12, 96), [None]),
            ('strong_smelling_urine', (4, 7), (12, 96), [None]),
            ('blood_in_urine', (6, 9), (12, 72), [None]),
            ('pelvic_pain', (5, 8), (12, 96), ['pelvis', 'lower_abdomen']),
            ('back_pain', (4, 7), (12, 96), ['lower_back']),
            ('feeling_tired', (4, 7), (24, 96), ['whole_body']),
            ('mild_fever', (3, 6), (12, 72), ['whole_body']),
            ('chills', (3, 6), (12, 48), ['whole_body'])
        ],
        'symptom_count': (4, 7),
        'age_range': (18, 70),
        'confidence_range': (0.70, 0.90)
    },
    'common_cold': {
        'symptoms': [
            ('runny_nose', (4, 7), (48, 168), ['nose']),
            ('nasal_congestion', (5, 8), (48, 168), ['nose', 'sinuses']),
            ('sneezing', (4, 7), (48, 120), ['nose']),
            ('sore_throat', (3, 6), (24, 96), ['throat']),
            ('scratchy_throat', (3, 6), (24, 96), ['throat']),
            ('mild_cough', (3, 7), (72, 240), ['chest', 'throat']),
            ('post_nasal_drip', (3, 6), (48, 168), ['throat']),
            ('watery_eyes', (3, 6), (24, 96), ['eyes']),
            ('mild_headache', (2, 5), (24, 96), ['whole_head']),
            ('mild_fatigue', (3, 6), (48, 168), ['whole_body']),
            ('low_grade_fever', (2, 4), (24, 72), ['whole_body']),
            ('slight_body_aches', (2, 5), (48, 120), ['muscles']),
            ('reduced_sense_taste', (3, 6), (48, 168), [None]),
            ('reduced_sense_smell', (3, 6), (48, 168), ['nose'])
        ],
        'symptom_count': (4, 8),
        'age_range': (1, 80),
        'confidence_range': (0.65, 0.85)
    },
    'sinusitis': {
        'symptoms': [
            ('facial_pain', (6, 9), (72, 336), ['cheeks', 'forehead', 'around_eyes']),
            ('facial_pressure', (6, 9), (72, 336), ['sinuses', 'face']),
            ('nasal_congestion', (7, 9), (72, 336), ['nose', 'sinuses']),
            ('thick_nasal_discharge', (6, 9), (72, 336), ['nose']),
            ('yellow_green_mucus', (5, 8), (72, 336), ['nose']),
            ('post_nasal_drip', (5, 8), (72, 336), ['throat']),
            ('reduced_smell', (5, 8), (72, 336), ['nose']),
            ('reduced_taste', (4, 7), (72, 336), [None]),
            ('cough', (4, 7), (72, 240), ['throat', 'chest']),
            ('headache', (5, 8), (48, 240), ['forehead', 'face']),
            ('ear_pressure', (4, 7), (48, 240), ['ears']),
            ('upper_tooth_pain', (5, 8), (48, 240), ['upper_teeth', 'jaw']),
            ('fever', (4, 7), (48, 168), ['whole_body']),
            ('fatigue', (5, 8), (72, 240), ['whole_body']),
            ('bad_breath', (4, 6), (72, 336), [None])
        ],
        'symptom_count': (5, 9),
        'age_range': (10, 70),
        'confidence_range': (0.70, 0.88)
    },
    'gastritis': {
        'symptoms': [
            ('burning_stomach', (6, 9), (6, 168), ['stomach', 'upper_abdomen']),
            ('gnawing_stomach_pain', (5, 9), (6, 168), ['stomach']),
            ('upper_abdominal_pain', (5, 8), (6, 168), ['upper_abdomen']),
            ('nausea', (5, 8), (6, 168), ['stomach']),
            ('vomiting', (5, 8), (6, 72), ['stomach']),
            ('bloating', (4, 7), (6, 168), ['abdomen']),
            ('feeling_full_quickly', (5, 7), (6, 168), ['stomach']),
            ('indigestion', (5, 8), (6, 168), ['stomach']),
            ('loss_appetite', (4, 7), (12, 168), [None]),
            ('belching', (3, 6), (6, 168), [None]),
            ('hiccups', (2, 5), (1, 24), [None]),
            ('black_tarry_stools', (6, 9), (12, 72), [None]),
            ('vomiting_blood', (7, 10), (6, 48), [None]),
            ('heartburn', (4, 7), (6, 168), ['chest', 'stomach'])
        ],
        'symptom_count': (4, 7),
        'age_range': (20, 70),
        'confidence_range': (0.65, 0.85)
    },
    'pneumonia': {
        'symptoms': [
            ('productive_cough', (7, 10), (72, 336), ['chest', 'lungs']),
            ('chest_pain_breathing', (7, 10), (48, 240), ['chest']),
            ('shortness_breath', (7, 10), (48, 240), ['lungs', 'chest']),
            ('high_fever', (7, 10), (48, 240), ['whole_body']),
            ('chills', (6, 9), (48, 168), ['whole_body']),
            ('sweating', (6, 9), (48, 168), ['whole_body']),
            ('fatigue', (7, 10), (72, 336), ['whole_body']),
            ('rapid_breathing', (6, 9), (48, 240), ['lungs']),
            ('rapid_heartbeat', (6, 9), (48, 240), ['chest']),
            ('confusion', (5, 8), (24, 168), [None]),
            ('nausea', (4, 7), (24, 120), [None]),
            ('vomiting', (4, 7), (24, 96), [None]),
            ('muscle_aches', (5, 8), (48, 168), ['muscles']),
            ('headache', (4, 7), (48, 168), ['whole_head'])
        ],
        'symptom_count': (5, 9),
        'age_range': (1, 90),
        'confidence_range': (0.75, 0.92)
    },
    'appendicitis': {
        'symptoms': [
            ('sudden_pain_around_navel', (7, 10), (6, 24), ['around_belly_button']),
            ('pain_moves_lower_right', (8, 10), (12, 36), ['lower_right_abdomen']),
            ('sharp_pain_lower_right', (8, 10), (12, 48), ['lower_right_abdomen']),
            ('pain_worsens_movement', (8, 10), (12, 48), ['lower_right_abdomen']),
            ('pain_worsens_coughing', (7, 10), (12, 48), ['lower_right_abdomen']),
            ('nausea', (6, 9), (6, 36), [None]),
            ('vomiting', (6, 9), (6, 36), [None]),
            ('loss_appetite', (6, 9), (12, 48), [None]),
            ('low_grade_fever', (5, 8), (12, 48), ['whole_body']),
            ('inability_pass_gas', (5, 8), (12, 48), ['intestines']),
            ('abdominal_swelling', (5, 8), (12, 48), ['abdomen']),
            ('constipation', (4, 7), (12, 72), [None]),
            ('diarrhea', (3, 6), (12, 48), [None])
        ],
        'symptom_count': (5, 8),
        'age_range': (10, 50),
        'confidence_range': (0.70, 0.92)
    },
    'asthma_attack': {
        'symptoms': [
            ('severe_shortness_breath', (8, 10), (0.5, 6), ['lungs', 'chest']),
            ('wheezing', (7, 10), (0.5, 6), ['chest', 'lungs']),
            ('chest_tightness', (7, 10), (0.5, 6), ['chest']),
            ('rapid_breathing', (7, 10), (0.5, 6), ['lungs']),
            ('persistent_cough', (6, 9), (1, 12), ['chest', 'throat']),
            ('difficulty_speaking', (6, 9), (0.5, 4), [None]),
            ('panic_feeling', (6, 9), (0.5, 4), [None]),
            ('pale_face', (5, 8), (0.5, 4), ['face']),
            ('blue_lips', (7, 10), (0.5, 2), ['lips']),
            ('fatigue', (6, 9), (2, 12), ['whole_body']),
            ('sweating', (5, 8), (0.5, 4), ['whole_body'])
        ],
        'symptom_count': (4, 7),
        'age_range': (5, 70),
        'confidence_range': (0.75, 0.93)
    },
    'kidney_stones': {
        'symptoms': [
            ('severe_side_pain', (9, 10), (1, 48), ['side', 'back', 'below_ribs']),
            ('pain_radiates_groin', (8, 10), (1, 48), ['lower_abdomen', 'groin']),
            ('waves_of_pain', (8, 10), (1, 48), ['side', 'back']),
            ('pain_urinating', (7, 10), (6, 72), ['urethra']),
            ('pink_red_urine', (6, 9), (6, 72), [None]),
            ('cloudy_urine', (5, 8), (6, 72), [None]),
            ('foul_smelling_urine', (5, 8), (6, 72), [None]),
            ('frequent_urination', (6, 9), (12, 72), ['bladder']),
            ('nausea', (6, 9), (1, 48), [None]),
            ('vomiting', (6, 9), (1, 48), [None]),
            ('fever', (5, 8), (12, 72), ['whole_body']),
            ('chills', (5, 8), (12, 72), ['whole_body'])
        ],
        'symptom_count': (5, 8),
        'age_range': (20, 60),
        'confidence_range': (0.75, 0.90)
    },
    'allergic_reaction': {
        'symptoms': [
            ('skin_rash', (5, 9), (0.25, 48), ['skin', 'body']),
            ('hives', (6, 9), (0.25, 48), ['skin']),
            ('itching', (6, 9), (0.5, 72), ['skin', 'whole_body']),
            ('swelling', (5, 9), (0.5, 48), ['face', 'lips', 'tongue', 'throat']),
            ('red_watery_eyes', (4, 7), (1, 48), ['eyes']),
            ('runny_nose', (4, 7), (1, 48), ['nose']),
            ('sneezing', (4, 7), (1, 48), ['nose']),
            ('nasal_congestion', (4, 7), (1, 48), ['nose']),
            ('wheezing', (6, 9), (0.5, 24), ['chest', 'lungs']),
            ('difficulty_breathing', (7, 10), (0.25, 12), ['lungs', 'throat']),
            ('throat_tightness', (6, 9), (0.25, 12), ['throat']),
            ('nausea', (4, 7), (0.5, 24), [None]),
            ('vomiting', (5, 8), (0.5, 24), [None]),
            ('diarrhea', (4, 7), (1, 48), [None]),
            ('dizziness', (5, 8), (0.25, 12), ['head'])
        ],
        'symptom_count': (4, 9),
        'age_range': (1, 80),
        'confidence_range': (0.65, 0.88)
    }
}

# Generate Table 1: Patient Cases
cases = []
for i in range(n_cases):
    diagnosis = np.random.choice(list(conditions.keys()))
    condition = conditions[diagnosis]
    
    age = np.random.randint(condition['age_range'][0], condition['age_range'][1] + 1)
    gender = np.random.choice(['M', 'F'], p=[0.48, 0.52])  # Slightly more female cases
    timestamp = datetime(2024, 1, 1) + timedelta(
        days=np.random.randint(0, 365),
        hours=np.random.randint(0, 24),
        minutes=np.random.randint(0, 60)
    )
    confidence = round(np.random.uniform(condition['confidence_range'][0], condition['confidence_range'][1]), 2)
    
    cases.append({
        'case_id': f'C{i+1:04d}',
        'user_id': f'U{np.random.randint(1, 400):04d}',
        'age': age,
        'gender': gender,
        'timestamp': timestamp,
        'diagnosis': diagnosis,
        'confidence': confidence
    })

df_cases = pd.DataFrame(cases)

# Generate Table 2: Symptoms per Case
symptoms_data = []
for _, case in df_cases.iterrows():
    diagnosis = case['diagnosis']
    condition = conditions[diagnosis]
    
    # Determine how many symptoms for this case (realistic variation)
    num_symptoms = np.random.randint(condition['symptom_count'][0], condition['symptom_count'][1] + 1)
    
    # Select symptoms with realistic probabilities
    # Primary symptoms more likely, secondary less likely
    all_symptoms = condition['symptoms']
    
    # Give higher probability to first few symptoms (primary symptoms)
    symptom_weights = np.array([1.0] * len(all_symptoms))
    symptom_weights[:3] = 2.0  # Primary symptoms twice as likely
    symptom_weights = symptom_weights / symptom_weights.sum()
    
    # Randomly select symptoms with weights
    selected_indices = np.random.choice(
        range(len(all_symptoms)),
        size=min(num_symptoms, len(all_symptoms)),
        replace=False,
        p=symptom_weights
    )
    
    for symptom_idx in selected_indices:
        symptom_name, severity_range, duration_range, locations = all_symptoms[symptom_idx]
        
        # Generate severity with some realistic skew (higher severity more common for serious symptoms)
        severity = np.random.randint(severity_range[0], severity_range[1] + 5)
        
        # Duration with some variation
        duration = round(np.random.uniform(duration_range[0], duration_range[1]), 1)
        
        # Select location
        location = np.random.choice(locations) if locations[0] is not None else None
        
        symptoms_data.append({
            'case_id': case['case_id'],
            'symptom': symptom_name,
            'severity': severity,
            'duration_hours': duration,
            'body_location': location
        })

df_symptoms = pd.DataFrame(symptoms_data)

# Display results
print("=" * 100)
print("TABLE 1: PATIENT CASES")
print("=" * 100)
print(f"Shape: {df_cases.shape}")
print(f"Columns: {list(df_cases.columns)}")
print("\nFirst 15 cases:")
print(df_cases.head(15).to_string(index=False))

print("\n\n" + "=" * 100)
print("TABLE 2: SYMPTOMS PER CASE")
print("=" * 100)
print(f"Shape: {df_symptoms.shape}")
print(f"Columns: {list(df_symptoms.columns)}")
print("\nFirst 30 symptom records:")
print(df_symptoms.head(30).to_string(index=False))

print("\n\n" + "=" * 100)
print("DATASET STATISTICS")
print("=" * 100)

print("\n📊 Diagnosis Distribution:")
diag_counts = df_cases['diagnosis'].value_counts()
for diag, count in diag_counts.items():
    print(f"  {diag:25s}: {count:4d} cases ({count/len(df_cases)*100:5.1f}%)")

print("\n📊 Symptoms Per Case:")
symptom_counts = df_symptoms.groupby('case_id').size()
print(f"  Mean:   {symptom_counts.mean():.2f} symptoms")
print(f"  Median: {symptom_counts.median():.1f} symptoms")
print(f"  Min:    {symptom_counts.min()} symptoms")
print(f"  Max:    {symptom_counts.max()} symptoms")

print("\n📊 Top 15 Most Common Symptoms:")
top_symptoms = df_symptoms['symptom'].value_counts().head(15)
for symptom, count in top_symptoms.items():
    print(f"  {symptom:30s}: {count:4d} occurrences")

print("\n📊 Age Distribution:")
print(f"  Mean:   {df_cases['age'].mean():.1f} years")
print(f"  Median: {df_cases['age'].median():.0f} years")
print(f"  Range:  {df_cases['age'].min()}-{df_cases['age'].max()} years")

print("\n📊 Gender Distribution:")
gender_counts = df_cases['gender'].value_counts()
for gender, count in gender_counts.items():
    print(f"  {gender}: {count:4d} cases ({count/len(df_cases)*100:5.1f}%)")

print("\n\n" + "=" * 100)
print("EXAMPLE: Complete Case with All Symptoms")
print("=" * 100)

# Show a few interesting cases
for sample_idx in [0, 50, 100]:
    sample_case_id = df_cases.iloc[sample_idx]['case_id']
    print(f"\n{'─' * 100}")
    print("CASE DETAILS:")
    case_info = df_cases[df_cases['case_id'] == sample_case_id]
    print(case_info.to_string(index=False))
    
    print(f"\nALL SYMPTOMS FOR THIS CASE:")
    case_symptoms = df_symptoms[df_symptoms['case_id'] == sample_case_id]
    print(case_symptoms.to_string(index=False))
    print(f"Total symptoms: {len(case_symptoms)}")

print("\n\n" + "=" * 100)
print("✅ REALISTIC MEDICAL DATASET GENERATED!")
print("=" * 100)
print(f"📋 {len(df_cases)} patient cases across {len(conditions)} diagnoses")
print(f"🩺 {len(df_symptoms)} total symptom records")
print(f"📊 Average {len(df_symptoms)/len(df_cases):.1f} symptoms per case")
print(f"🔬 {len(df_symptoms['symptom'].unique())} unique symptom types")
print(f"📍 {df_symptoms['body_location'].nunique()} unique body locations")
print("=" * 100)

# Optionally save to CSV
# df_cases.to_csv('patient_cases.csv', index=False)
# df_symptoms.to_csv('symptoms_per_case.csv', index=False)

TABLE 1: PATIENT CASES
Shape: (1000, 7)
Columns: ['case_id', 'user_id', 'age', 'gender', 'timestamp', 'diagnosis', 'confidence']

First 15 cases:
case_id user_id  age gender           timestamp        diagnosis  confidence
  C0001   U0215   69      F 2024-04-16 07:20:00              uti        0.73
  C0002   U0309   75      F 2024-04-09 07:23:00        pneumonia        0.86
  C0003   U0022   34      M 2024-07-10 20:32:00        influenza        0.85
  C0004   U0175   53      F 2024-06-18 15:14:00    asthma_attack        0.83
  C0005   U0388   22      F 2024-05-14 20:08:00    kidney_stones        0.76
  C0006   U0367   69      F 2024-09-21 20:01:00        sinusitis        0.82
  C0007   U0050   16      F 2024-09-20 14:34:00     appendicitis        0.90
  C0008   U0191    4      M 2024-02-23 09:03:00      common_cold        0.69
  C0009   U0215   48      F 2024-09-26 15:14:00        influenza        0.81
  C0010   U0217   49      F 2024-08-24 17:46:00     appendicitis        0.88
  C0011

In [11]:
class MedicalDiagnosisCleaner:
    """
    Handles all data cleaning and preparation steps for medical diagnosis prediction.
    Transforms normalized database tables into ML-ready feature matrices.
    """
    
    def __init__(self):
        """Initialize the data cleaner."""
        self.feature_columns = []
        self.symptom_columns = []
        self.metadata = {}
        self.df_prepared = None
        
    def clean_and_prepare(self, df_cases: pd.DataFrame, df_symptoms: pd.DataFrame) -> pd.DataFrame:
        """
        Main pipeline: clean and prepare data from normalized tables.
        
        Args:
            df_cases: DataFrame with patient cases
            df_symptoms: DataFrame with symptoms per case
            
        Returns:
            df_prepared: Cleaned DataFrame ready for ML
        """
        print("=" * 100)
        print("MEDICAL DIAGNOSIS DATA CLEANING PIPELINE")
        print("=" * 100)
        
        # Step 1: Validate input data
        print("\n[1/9] Validating input data...")
        self._validate_data(df_cases, df_symptoms)
        print("   ✓ Data validation passed")
        
        # Step 2: Clean cases table
        print("\n[2/9] Cleaning cases table...")
        df_cases_clean = self._clean_cases(df_cases)
        print(f"   ✓ Cases cleaned: {df_cases_clean.shape}")
        
        # Step 3: Clean symptoms table
        print("\n[3/9] Cleaning symptoms table...")
        df_symptoms_clean = self._clean_symptoms(df_symptoms)
        print(f"   ✓ Symptoms cleaned: {df_symptoms_clean.shape}")
        
        # Step 4: Pivot symptoms to wide format
        print("\n[4/9] Pivoting symptoms to wide format...")
        df_severity = self._pivot_symptoms(df_symptoms_clean)
        print(f"   ✓ Symptom matrix created: {df_severity.shape}")
        print(f"   ✓ Unique symptoms: {len(df_severity.columns)}")
        
        # Step 5: Engineer aggregate features
        print("\n[5/9] Engineering aggregate features...")
        df_features = self._engineer_features(df_symptoms_clean)
        print(f"   ✓ Aggregate features created: {len(df_features.columns) - 1}")
        
        # Step 6: Merge all data
        print("\n[6/9] Merging all features...")
        df_merged = self._merge_all(df_cases_clean, df_features, df_severity)
        print(f"   ✓ Merged dataset shape: {df_merged.shape}")
        
        # Step 7: Handle missing values
        print("\n[7/9] Handling missing values...")
        df_complete = self._handle_missing_values(df_merged)
        print(f"   ✓ Missing values handled: {df_complete.shape}")
        
        # Step 8: Encode categorical variables
        print("\n[8/9] Encoding categorical variables...")
        df_encoded = self._encode_categoricals(df_complete)
        print(f"   ✓ Encoding complete: {df_encoded.shape}")
        
        # Step 9: Remove useless fields
        print("\n[9/9] Removing useless fields...")
        df_prepared = self._remove_useless_fields(df_encoded)
        print(f"   ✓ Useless fields removed: {df_prepared.shape}")
        
        # Store the prepared dataframe
        self.df_prepared = df_prepared
        
        # Store metadata
        self._store_metadata(df_prepared, df_cases, df_symptoms)
        
        print("\n" + "=" * 100)
        print("DATA CLEANING COMPLETE ✅")
        print("=" * 100)
        self._print_summary()
        
        return df_prepared
    
    def _validate_data(self, df_cases: pd.DataFrame, df_symptoms: pd.DataFrame) -> None:
        """Validate that input data has required columns and structure."""
        # Check cases table
        required_case_cols = ['case_id', 'user_id', 'age', 'gender', 'timestamp', 'diagnosis', 'confidence']
        missing_case_cols = [col for col in required_case_cols if col not in df_cases.columns]
        if missing_case_cols:
            raise ValueError(f"Cases table missing columns: {missing_case_cols}")
        
        # Check symptoms table
        required_symptom_cols = ['case_id', 'symptom', 'severity', 'duration_hours']
        missing_symptom_cols = [col for col in required_symptom_cols if col not in df_symptoms.columns]
        if missing_symptom_cols:
            raise ValueError(f"Symptoms table missing columns: {missing_symptom_cols}")
        
        # Check for orphaned symptoms (symptoms without cases)
        orphaned = set(df_symptoms['case_id'].unique()) - set(df_cases['case_id'].unique())
        if orphaned:
            print(f"   ⚠ Warning: {len(orphaned)} orphaned symptom records found")
    
    def _clean_cases(self, df_cases: pd.DataFrame) -> pd.DataFrame:
        """Clean the cases table."""
        df = df_cases.copy()
        
        # Remove duplicates
        initial_count = len(df)
        df = df.drop_duplicates(subset=['case_id'])
        if len(df) < initial_count:
            print(f"   ⚠ Removed {initial_count - len(df)} duplicate cases")
        
        # Clean age (remove invalid values)
        df = df[(df['age'] > 0) & (df['age'] < 120)]
        
        # Clean gender
        df['gender'] = df['gender'].str.upper()
        df = df[df['gender'].isin(['M', 'F'])]
        
        # Clean confidence
        df['confidence'] = df['confidence'].clip(0, 1)
        
        # Convert timestamp to datetime if not already
        if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        return df
    
    def _clean_symptoms(self, df_symptoms: pd.DataFrame) -> pd.DataFrame:
        """Clean the symptoms table."""
        df = df_symptoms.copy()
        
        # Remove duplicates (same case + same symptom)
        initial_count = len(df)
        df = df.drop_duplicates(subset=['case_id', 'symptom'])
        if len(df) < initial_count:
            print(f"   ⚠ Removed {initial_count - len(df)} duplicate symptoms")
        
        # Clean symptom names (lowercase, remove extra spaces)
        df['symptom'] = df['symptom'].str.lower().str.strip().str.replace(r'\s+', '_', regex=True)
        
        # Clean severity (1-10 range)
        df['severity'] = df['severity'].clip(1, 10)
        
        # Clean duration (remove negative values)
        df['duration_hours'] = df['duration_hours'].clip(0, None)
        
        # Clean body_location (lowercase, handle NaN)
        if 'body_location' in df.columns:
            df['body_location'] = df['body_location'].fillna('unknown')
            df['body_location'] = df['body_location'].str.lower().str.strip()
        
        return df
    
    def _pivot_symptoms(self, df_symptoms: pd.DataFrame) -> pd.DataFrame:
        """
        Pivot symptoms to wide format: one column per symptom with severity values.
        """
        df_severity = df_symptoms.pivot_table(
            index='case_id',
            columns='symptom',
            values='severity',
            fill_value=0,
            aggfunc='max'  # If duplicate, take max severity
        )
        
        # Store symptom column names
        self.symptom_columns = list(df_severity.columns)
        
        return df_severity
    
    def _engineer_features(self, df_symptoms: pd.DataFrame) -> pd.DataFrame:
        """
        Engineer aggregate features from symptoms.
        
        Returns:
            DataFrame with case_id and engineered features
        """
        features = {}
        
        # Group by case_id
        grouped = df_symptoms.groupby('case_id')
        
        # Feature 1: Total symptom count
        features['symptom_count'] = grouped.size()
        
        # Feature 2: Average severity
        features['avg_severity'] = grouped['severity'].mean()
        
        # Feature 3: Max severity
        features['max_severity'] = grouped['severity'].max()
        
        # Feature 4: Min severity
        features['min_severity'] = grouped['severity'].min()
        
        # Feature 5: Severity standard deviation
        features['std_severity'] = grouped['severity'].std().fillna(0)
        
        # Feature 6: Average duration
        features['avg_duration'] = grouped['duration_hours'].mean()
        
        # Feature 7: Max duration
        features['max_duration'] = grouped['duration_hours'].max()
        
        # Feature 8: Min duration
        features['min_duration'] = grouped['duration_hours'].min()
        
        # Feature 9: Total duration (sum of all symptoms)
        features['total_duration'] = grouped['duration_hours'].sum()
        
        # Feature 10: Has fever (binary)
        fever_symptoms = df_symptoms[df_symptoms['symptom'].str.contains('fever', case=False, na=False)]
        has_fever = fever_symptoms.groupby('case_id').size()
        features['has_fever'] = (has_fever > 0).astype(int)
        
        # Feature 11: Has pain (binary)
        pain_symptoms = df_symptoms[df_symptoms['symptom'].str.contains('pain|ache', case=False, na=False)]
        has_pain = pain_symptoms.groupby('case_id').size()
        features['has_pain'] = (has_pain > 0).astype(int)
        
        # Feature 12: Has nausea/vomiting (binary)
        gi_symptoms = df_symptoms[df_symptoms['symptom'].str.contains('nausea|vomit', case=False, na=False)]
        has_gi = gi_symptoms.groupby('case_id').size()
        features['has_gi_symptoms'] = (has_gi > 0).astype(int)
        
        # Feature 13: Has respiratory symptoms (binary)
        resp_symptoms = df_symptoms[df_symptoms['symptom'].str.contains('cough|breath|wheez', case=False, na=False)]
        has_resp = resp_symptoms.groupby('case_id').size()
        features['has_respiratory'] = (has_resp > 0).astype(int)
        
        # Combine all features into a DataFrame
        df_features = pd.DataFrame(features).reset_index()
        
        return df_features
    
    def _merge_all(self, df_cases: pd.DataFrame, df_features: pd.DataFrame, 
                   df_severity: pd.DataFrame) -> pd.DataFrame:
        """Merge cases, engineered features, and symptom matrix."""
        # Start with cases
        df = df_cases.copy()
        
        # Merge engineered features
        df = df.merge(df_features, on='case_id', how='left')
        
        # Merge symptom severity matrix
        df = df.merge(df_severity, left_on='case_id', right_index=True, how='left')
        
        return df
    
    def _handle_missing_values(self, df: pd.DataFrame) -> pd.DataFrame:
        """Handle missing values in the dataset."""
        df = df.copy()
        
        # Fill numeric columns with 0
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(0)
        
        # Fill categorical/string columns with 'unknown'
        # Use exclude to avoid the warning
        categorical_cols = df.select_dtypes(exclude=[np.number, 'datetime64']).columns
        df[categorical_cols] = df[categorical_cols].fillna('unknown')
        
        return df
    
    def _encode_categoricals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Encode categorical variables."""
        df = df.copy()
        
        # Encode gender (M=0, F=1) - REPLACE the original gender column
        if 'gender' in df.columns:
            df['gender'] = (df['gender'] == 'F').astype(int)
            print("   ✓ Gender encoded: M=0, F=1")
        
        # Extract time features from timestamp
        if 'timestamp' in df.columns and pd.api.types.is_datetime64_any_dtype(df['timestamp']):
            df['hour_of_day'] = df['timestamp'].dt.hour
            df['day_of_week'] = df['timestamp'].dt.dayofweek
            df['month'] = df['timestamp'].dt.month
            print("   ✓ Time features extracted: hour_of_day, day_of_week, month")
        
        return df
    
    def _remove_useless_fields(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Remove fields not needed for ML training (case_id, user_id, timestamp, confidence).
        This is now part of the cleaning pipeline.
        """
        df = df.copy()
        
        # Fields to remove
        useless_fields = ['case_id', 'user_id', 'timestamp', 'confidence']
        fields_to_drop = [col for col in useless_fields if col in df.columns]
        
        if fields_to_drop:
            df = df.drop(columns=fields_to_drop)
            print(f"   ✓ Removed: {', '.join(fields_to_drop)}")
        
        return df
    
    def _store_metadata(self, df_prepared: pd.DataFrame, df_cases: pd.DataFrame, 
                       df_symptoms: pd.DataFrame) -> None:
        """Store metadata about the cleaning process."""
        self.metadata = {
            'original_cases': len(df_cases),
            'original_symptoms': len(df_symptoms),
            'final_cases': len(df_prepared),
            'total_features': df_prepared.shape[1],
            'symptom_features': len(self.symptom_columns),
            'unique_symptoms': self.symptom_columns,
            'diagnoses': list(df_prepared['diagnosis'].unique()),
            'n_diagnoses': df_prepared['diagnosis'].nunique(),
            'age_range': (df_prepared['age'].min(), df_prepared['age'].max()),
            'timestamp': datetime.now().isoformat()
        }
    
    def _print_summary(self) -> None:
        """Print summary of cleaned data."""
        print(f"\n📊 CLEANING SUMMARY:")
        print(f"   • Original cases:     {self.metadata['original_cases']}")
        print(f"   • Final cases:        {self.metadata['final_cases']}")
        print(f"   • Total features:     {self.metadata['total_features']}")
        print(f"   • Symptom features:   {self.metadata['symptom_features']}")
        print(f"   • Unique diagnoses:   {self.metadata['n_diagnoses']}")
        print(f"   • Age range:          {self.metadata['age_range'][0]}-{self.metadata['age_range'][1]} years")
    
    def get_feature_columns(self) -> List[str]:
        """
        Get list of feature columns (excludes diagnosis which is the target).
        
        Returns:
            List of feature column names
        """
        if self.df_prepared is None:
            raise ValueError("No data prepared yet. Call clean_and_prepare() first.")
        
        # Exclude only diagnosis (target variable)
        feature_cols = [col for col in self.df_prepared.columns if col != 'diagnosis']
        
        return feature_cols
    
    def get_X_y(self) -> Tuple[pd.DataFrame, pd.Series]:
        """
        Get feature matrix (X) and target vector (y).
        
        Returns:
            X: Features DataFrame
            y: Target Series (diagnosis - NOT encoded yet)
        """
        if self.df_prepared is None:
            raise ValueError("No data prepared yet. Call clean_and_prepare() first.")
        
        feature_cols = self.get_feature_columns()
        X = self.df_prepared[feature_cols]
        y = self.df_prepared['diagnosis']
        
        return X, y
    
    def save_prepared_data(self, filepath: str) -> None:
        """Save prepared data to CSV."""
        if self.df_prepared is None:
            raise ValueError("No data prepared yet. Call clean_and_prepare() first.")
        
        self.df_prepared.to_csv(filepath, index=False)
        print(f"\n💾 Prepared data saved to: {filepath}")
    
    def get_metadata(self) -> Dict:
        """Get metadata about the cleaning process."""
        return self.metadata

In [12]:
# Assuming df_cases and df_symptoms are already generated
# from your earlier code

print("\n" + "🏥" * 50)
print("MEDICAL DIAGNOSIS DATA CLEANING")
print("🏥" * 50)

# Initialize cleaner
cleaner = MedicalDiagnosisCleaner()

# Clean and prepare data
df_prepared = cleaner.clean_and_prepare(df_cases, df_symptoms)

# Get feature matrix and target
X, y = cleaner.get_X_y()

print(f"\n✅ Ready for ML training!")
print(f"   • Features (X): {X.shape}")
print(f"   • Target (y):   {y.shape}")

# Get feature columns
feature_columns = cleaner.get_feature_columns()
print(f"\n📋 Feature columns ({len(feature_columns)}):")
for i, col in enumerate(feature_columns[:20], 1):
    print(f"   {i:2d}. {col}")
if len(feature_columns) > 20:
    print(f"   ... and {len(feature_columns) - 20} more")

# Get metadata
metadata = cleaner.get_metadata()
print(f"\n📊 Metadata:")
print(f"   • Diagnoses: {', '.join(metadata['diagnoses'][:5])}...")

# Save prepared data (optional)
# cleaner.save_prepared_data('prepared_medical_data.csv')


🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥
MEDICAL DIAGNOSIS DATA CLEANING
🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥
MEDICAL DIAGNOSIS DATA CLEANING PIPELINE

[1/9] Validating input data...
   ✓ Data validation passed

[2/9] Cleaning cases table...
   ✓ Cases cleaned: (1000, 7)

[3/9] Cleaning symptoms table...
   ✓ Symptoms cleaned: (6183, 5)

[4/9] Pivoting symptoms to wide format...
   ✓ Symptom matrix created: (1000, 138)
   ✓ Unique symptoms: 138

[5/9] Engineering aggregate features...
   ✓ Aggregate features created: 13

[6/9] Merging all features...
   ✓ Merged dataset shape: (1000, 158)

[7/9] Handling missing values...
   ✓ Missing values handled: (1000, 158)

[8/9] Encoding categorical variables...
   ✓ Gender encoded: M=0, F=1
   ✓ Time features extracted: hour_of_day, day_of_week, month
   ✓ Encoding complete: (1000, 161)

[9/9] Removing useless fields...
   ✓ Removed: case_id, user_id, timestamp, confidence
   ✓ Useless fields removed: (1000, 157)

DATA 

In [13]:
warnings.filterwarnings('ignore')

class MedicalDiagnosisPredictor:
    """
    Handles ML model training, evaluation, comparison, and prediction for medical diagnosis.
    Works with data prepared by MedicalDiagnosisCleaner.
    """
    
    def __init__(self):
        """Initialize the predictor."""
        self.models = {}
        self.best_model = None
        self.best_model_name = None
        self.label_encoder = LabelEncoder()
        self.scaler = StandardScaler()
        self.feature_names = []
        self.is_trained = False
        self.training_history = []
        self.comparison_results = pd.DataFrame()
        
    def prepare_data(self, X: pd.DataFrame, y: pd.Series, 
                    test_size: float = 0.2, 
                    random_state: int = 42) -> None:
        """
        Prepare data for training: encode labels, split, and scale.
        
        Args:
            X: Feature matrix (from cleaner.get_X_y())
            y: Target variable (diagnosis names)
            test_size: Proportion for test set (default 0.2 = 20%)
            random_state: Random seed for reproducibility
        """
        print("=" * 100)
        print("PREPARING DATA FOR MODEL TRAINING")
        print("=" * 100)
        
        # Store feature names
        self.feature_names = list(X.columns)
        
        # Step 1: Encode target labels
        print(f"\n[1/3] Encoding target labels...")
        self.y_encoded = self.label_encoder.fit_transform(y)
        print(f"   ✓ Encoded {len(self.label_encoder.classes_)} classes")
        
        # Display class mapping
        print(f"\n   Class Mapping:")
        for idx, diagnosis in enumerate(self.label_encoder.classes_):
            count = (self.y_encoded == idx).sum()
            print(f"      {idx} = {diagnosis:20s} ({count} cases)")
        
        # Step 2: Split data
        print(f"\n[2/3] Splitting data (train/test = {100*(1-test_size):.0f}/{100*test_size:.0f})...")
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, self.y_encoded,
            test_size=test_size,
            random_state=random_state,
            stratify=self.y_encoded
        )
        print(f"   ✓ Training set: {self.X_train.shape[0]} samples")
        print(f"   ✓ Test set:     {self.X_test.shape[0]} samples")
        print(f"   ✓ Features:     {self.X_train.shape[1]}")
        
        # Step 3: Scale features
        print(f"\n[3/3] Scaling features...")
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)
        print(f"   ✓ Features scaled (mean≈0, std≈1)")
        
        print("\n" + "=" * 100)
        print("DATA PREPARATION COMPLETE ✅")
        print("=" * 100)
    
    def train_single_model(self, model_name: str, model_params: Optional[Dict] = None) -> Dict:
        """
        Train a single model.
        
        Args:
            model_name: Name of model ('random_forest', 'gradient_boosting', etc.)
            model_params: Optional dict of model hyperparameters
            
        Returns:
            Dictionary with training results
        """
        if not hasattr(self, 'X_train_scaled'):
            raise ValueError("Data not prepared. Call prepare_data() first.")
        
        print(f"\n{'=' * 100}")
        print(f"TRAINING: {model_name.upper().replace('_', ' ')}")
        print(f"{'=' * 100}")
        
        # Initialize model
        model = self._get_model(model_name, model_params)
        
        # Train
        print(f"\n⏳ Training {model_name}...")
        start_time = datetime.now()
        model.fit(self.X_train_scaled, self.y_train)
        training_time = (datetime.now() - start_time).total_seconds()
        print(f"✅ Training complete in {training_time:.2f} seconds")
        
        # Predictions
        y_train_pred = model.predict(self.X_train_scaled)
        y_test_pred = model.predict(self.X_test_scaled)
        
        # Metrics
        train_accuracy = accuracy_score(self.y_train, y_train_pred)
        test_accuracy = accuracy_score(self.y_test, y_test_pred)
        
        results = {
            'model_name': model_name,
            'model': model,
            'train_accuracy': train_accuracy,
            'test_accuracy': test_accuracy,
            'training_time': training_time,
            'params': model_params or {},
            'y_train_pred': y_train_pred,
            'y_test_pred': y_test_pred,
            'timestamp': datetime.now().isoformat()
        }
        
        # Store model
        self.models[model_name] = results
        self.training_history.append(results)
        
        print(f"\n📊 Results:")
        print(f"   • Training Accuracy:   {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
        print(f"   • Test Accuracy:       {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
        print(f"   • Overfitting Gap:     {(train_accuracy - test_accuracy):.4f}")
        
        return results
    
    def train_multiple_models(self, model_configs: Optional[Dict] = None) -> pd.DataFrame:
        """
        Train multiple models and compare results.
        
        Args:
            model_configs: Optional dict of {model_name: params}
                          If None, uses default configurations
            
        Returns:
            DataFrame with comparison results
        """
        if model_configs is None:
            model_configs = self._get_default_model_configs()
        
        print("=" * 100)
        print(f"TRAINING {len(model_configs)} MODELS")
        print("=" * 100)
        
        results_list = []
        
        for model_name, params in model_configs.items():
            try:
                result = self.train_single_model(model_name, params)
                results_list.append({
                    'Model': model_name,
                    'Train Accuracy': result['train_accuracy'],
                    'Test Accuracy': result['test_accuracy'],
                    'Overfitting': result['train_accuracy'] - result['test_accuracy'],
                    'Training Time (s)': result['training_time']
                })
            except Exception as e:
                print(f"\n❌ Error training {model_name}: {e}")
                continue
        
        # Create comparison DataFrame
        df_comparison = pd.DataFrame(results_list)
        df_comparison = df_comparison.sort_values('Test Accuracy', ascending=False)
        self.comparison_results = df_comparison
        
        # Identify best model
        best_idx = df_comparison['Test Accuracy'].idxmax()
        self.best_model_name = df_comparison.loc[best_idx, 'Model']
        self.best_model = self.models[self.best_model_name]['model']
        self.is_trained = True
        
        print("\n" + "=" * 100)
        print("MODEL COMPARISON RESULTS")
        print("=" * 100)
        print(df_comparison.to_string(index=False))
        print("\n" + "=" * 100)
        print(f"🏆 BEST MODEL: {self.best_model_name.upper().replace('_', ' ')}")
        print(f"   Test Accuracy: {df_comparison.loc[best_idx, 'Test Accuracy']:.4f} ({df_comparison.loc[best_idx, 'Test Accuracy']*100:.2f}%)")
        print("=" * 100)
        
        return df_comparison
    
    def evaluate_model(self, model_name: Optional[str] = None, verbose: bool = True) -> Dict:
        """
        Detailed evaluation of a specific model.
        
        Args:
            model_name: Which model to evaluate (None = best model)
            verbose: Print detailed results
            
        Returns:
            Dictionary with evaluation metrics
        """
        if model_name is None:
            if self.best_model is None:
                raise ValueError("No models trained yet.")
            model_name = self.best_model_name
        
        if model_name not in self.models:
            raise ValueError(f"Model '{model_name}' not found. Available: {list(self.models.keys())}")
        
        model_results = self.models[model_name]
        y_test_pred = model_results['y_test_pred']
        
        if verbose:
            print("=" * 100)
            print(f"DETAILED EVALUATION: {model_name.upper().replace('_', ' ')}")
            print("=" * 100)
        
        # Calculate metrics
        accuracy = accuracy_score(self.y_test, y_test_pred)
        precision = precision_score(self.y_test, y_test_pred, average='weighted', zero_division=0)
        recall = recall_score(self.y_test, y_test_pred, average='weighted', zero_division=0)
        f1 = f1_score(self.y_test, y_test_pred, average='weighted', zero_division=0)
        
        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }
        
        if verbose:
            print(f"\n📊 Overall Metrics:")
            print(f"   • Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
            print(f"   • Precision: {precision:.4f}")
            print(f"   • Recall:    {recall:.4f}")
            print(f"   • F1-Score:  {f1:.4f}")
            
            # Classification report
            print(f"\n📋 Classification Report:")
            print("=" * 100)
            report = classification_report(
                self.y_test, 
                y_test_pred,
                target_names=self.label_encoder.classes_,
                digits=3
            )
            print(report)
            
            # Confusion matrix summary
            cm = confusion_matrix(self.y_test, y_test_pred)
            print(f"\n🔢 Confusion Matrix Summary:")
            print(f"   • Total predictions: {len(self.y_test)}")
            print(f"   • Correct: {np.trace(cm)} ({np.trace(cm)/len(self.y_test)*100:.1f}%)")
            print(f"   • Incorrect: {len(self.y_test) - np.trace(cm)} ({(len(self.y_test) - np.trace(cm))/len(self.y_test)*100:.1f}%)")
        
        return metrics
    
    def get_feature_importance(self, model_name: Optional[str] = None, top_n: int = 20) -> pd.DataFrame:
        """
        Get feature importance for tree-based models.
        
        Args:
            model_name: Which model (None = best model)
            top_n: Number of top features to return
            
        Returns:
            DataFrame with feature importances
        """
        if model_name is None:
            model_name = self.best_model_name
        
        if model_name not in self.models:
            raise ValueError(f"Model '{model_name}' not found.")
        
        model = self.models[model_name]['model']
        
        # Check if model has feature_importances_
        if not hasattr(model, 'feature_importances_'):
            print(f"⚠️ {model_name} does not have feature importance")
            return None
        
        # Create DataFrame
        importance_df = pd.DataFrame({
            'Feature': self.feature_names,
            'Importance': model.feature_importances_
        }).sort_values('Importance', ascending=False).head(top_n)
        
        print(f"\n🔝 Top {top_n} Most Important Features ({model_name}):")
        print("=" * 100)
        print(importance_df.to_string(index=False))
        
        return importance_df
    
    def predict(self, X_new: pd.DataFrame, return_proba: bool = True, 
               model_name: Optional[str] = None) -> Dict:
        """
        Make predictions on new data.
        
        Args:
            X_new: New feature data
            return_proba: Return probability distributions
            model_name: Which model to use (None = best model)
            
        Returns:
            Dictionary with predictions and probabilities
        """
        if not self.is_trained:
            raise ValueError("No models trained yet. Call train_multiple_models() first.")
        
        if model_name is None:
            model = self.best_model
            model_name = self.best_model_name
        else:
            if model_name not in self.models:
                raise ValueError(f"Model '{model_name}' not found.")
            model = self.models[model_name]['model']
        
        # Scale features
        X_scaled = self.scaler.transform(X_new)
        
        # Predict
        y_pred_encoded = model.predict(X_scaled)
        y_pred = self.label_encoder.inverse_transform(y_pred_encoded)
        
        results = {
            'predictions': y_pred,
            'predictions_encoded': y_pred_encoded,
            'model_used': model_name
        }
        
        # Probabilities
        if return_proba and hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_scaled)
            results['probabilities'] = y_proba
            results['top_3_predictions'] = []
            
            for i in range(len(X_new)):
                top_3_idx = np.argsort(y_proba[i])[::-1][:3]
                top_3 = [(self.label_encoder.classes_[idx], y_proba[i][idx]) 
                        for idx in top_3_idx]
                results['top_3_predictions'].append(top_3)
        
        return results
    
    def cross_validate_model(self, model_name: str, cv_folds: int = 5) -> Dict:
        """
        Perform cross-validation on a model.
        
        Args:
            model_name: Name of model to cross-validate
            cv_folds: Number of cross-validation folds
            
        Returns:
            Dictionary with CV results
        """
        print(f"\n{'=' * 100}")
        print(f"CROSS-VALIDATION: {model_name.upper().replace('_', ' ')} ({cv_folds} folds)")
        print(f"{'=' * 100}")
        
        model = self._get_model(model_name)
        
        # Combine train and test for CV
        X_all = np.vstack([self.X_train_scaled, self.X_test_scaled])
        y_all = np.concatenate([self.y_train, self.y_test])
        
        print(f"\n⏳ Running {cv_folds}-fold cross-validation...")
        cv_scores = cross_val_score(model, X_all, y_all, cv=cv_folds, scoring='accuracy')
        
        results = {
            'cv_scores': cv_scores,
            'mean_score': cv_scores.mean(),
            'std_score': cv_scores.std(),
            'min_score': cv_scores.min(),
            'max_score': cv_scores.max()
        }
        
        print(f"\n📊 Cross-Validation Results:")
        print(f"   • Mean Accuracy: {results['mean_score']:.4f} (±{results['std_score']:.4f})")
        print(f"   • Min Accuracy:  {results['min_score']:.4f}")
        print(f"   • Max Accuracy:  {results['max_score']:.4f}")
        print(f"   • All Scores:    {[f'{s:.4f}' for s in cv_scores]}")
        
        return results
    
    def tune_hyperparameters(self, model_name: str, param_grid: Dict, cv_folds: int = 3) -> Dict:
        """
        Perform hyperparameter tuning using GridSearchCV.
        
        Args:
            model_name: Name of model to tune
            param_grid: Dictionary of parameters to search
            cv_folds: Number of CV folds
            
        Returns:
            Dictionary with best parameters and results
        """
        print(f"\n{'=' * 100}")
        print(f"HYPERPARAMETER TUNING: {model_name.upper().replace('_', ' ')}")
        print(f"{'=' * 100}")
        
        model = self._get_model(model_name)
        
        print(f"\n⏳ Searching parameters...")
        print(f"   Parameter grid: {param_grid}")
        
        grid_search = GridSearchCV(
            model, param_grid,
            cv=cv_folds,
            scoring='accuracy',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(self.X_train_scaled, self.y_train)
        
        results = {
            'best_params': grid_search.best_params_,
            'best_score': grid_search.best_score_,
            'best_model': grid_search.best_estimator_
        }
        
        print(f"\n✅ Tuning Complete!")
        print(f"\n🎯 Best Parameters:")
        for param, value in results['best_params'].items():
            print(f"   • {param}: {value}")
        print(f"\n📊 Best CV Score: {results['best_score']:.4f}")
        
        # Train with best params
        print(f"\n⏳ Training with best parameters...")
        self.train_single_model(model_name, results['best_params'])
        
        return results
    
    def save_model(self, filepath: str, model_name: Optional[str] = None) -> None:
        """
        Save trained model to disk.
        
        Args:
            filepath: Where to save the model
            model_name: Which model to save (None = best model)
        """
        if not self.is_trained:
            raise ValueError("No models trained yet.")
        
        if model_name is None:
            model_name = self.best_model_name
        
        if model_name not in self.models:
            raise ValueError(f"Model '{model_name}' not found.")
        
        save_data = {
            'model': self.models[model_name]['model'],
            'label_encoder': self.label_encoder,
            'scaler': self.scaler,
            'feature_names': self.feature_names,
            'model_name': model_name,
            'train_accuracy': self.models[model_name]['train_accuracy'],
            'test_accuracy': self.models[model_name]['test_accuracy'],
            'timestamp': datetime.now().isoformat()
        }
        
        joblib.dump(save_data, filepath)
        print(f"\n💾 Model saved to: {filepath}")
        print(f"   • Model: {model_name}")
        print(f"   • Test Accuracy: {save_data['test_accuracy']:.4f}")
    
    def load_model(self, filepath: str) -> None:
        """
        Load a saved model from disk.
        
        Args:
            filepath: Path to saved model
        """
        print(f"\n📂 Loading model from: {filepath}")
        
        loaded_data = joblib.load(filepath)
        
        self.best_model = loaded_data['model']
        self.label_encoder = loaded_data['label_encoder']
        self.scaler = loaded_data['scaler']
        self.feature_names = loaded_data['feature_names']
        self.best_model_name = loaded_data['model_name']
        self.is_trained = True
        
        print(f"✅ Model loaded successfully!")
        print(f"   • Model: {self.best_model_name}")
        print(f"   • Test Accuracy: {loaded_data['test_accuracy']:.4f}")
    
    def _get_model(self, model_name: str, params: Optional[Dict] = None):
        """Initialize a model with given parameters."""
        if params is None:
            params = {}
        
        # Fixed: Each model gets its own correct class
        if model_name == 'random_forest':
            return RandomForestClassifier(random_state=42, n_jobs=-1, **params)
        elif model_name == 'gradient_boosting':
            return GradientBoostingClassifier(random_state=42, **params)
        elif model_name == 'logistic_regression':
            return LogisticRegression(random_state=42, max_iter=1000, **params)
        elif model_name == 'svm':
            return SVC(random_state=42, probability=True, **params)
        elif model_name == 'knn':
            return KNeighborsClassifier(**params)
        elif model_name == 'naive_bayes':
            return GaussianNB(**params)
        else:
            raise ValueError(f"Unknown model: {model_name}. Available: ['random_forest', 'gradient_boosting', 'logistic_regression', 'svm', 'knn', 'naive_bayes']")
        
    def _get_default_model_configs(self) -> Dict:
        """Get default configurations for multiple models."""
        return {
            'random_forest': {
                'n_estimators': 100,
                'max_depth': 15,
                'min_samples_split': 5,
                'min_samples_leaf': 2
            },
            'gradient_boosting': {
                'n_estimators': 100,
                'learning_rate': 0.1,
                'max_depth': 5
            },
            'logistic_regression': {
                'C': 1.0,
                'penalty': 'l2'
            },
            'knn': {
                'n_neighbors': 5,
                'weights': 'distance'
            },
            'naive_bayes': {}
        }
    
    def get_training_summary(self) -> Dict:
        """Get summary of all training."""
        if not self.training_history:
            return {'message': 'No models trained yet'}
        
        summary = {
            'total_models_trained': len(self.training_history),
            'best_model': self.best_model_name,
            'best_test_accuracy': self.models[self.best_model_name]['test_accuracy'],
            'all_models': list(self.models.keys()),
            'comparison_table': self.comparison_results.to_dict() if not self.comparison_results.empty else {}
        }
        
        return summary

In [14]:
# Initialize predictor
predictor = MedicalDiagnosisPredictor()

# Get X and y from your cleaned data
X, y = cleaner.get_X_y()

# Prepare data (encode, split, scale)
predictor.prepare_data(X, y, test_size=0.2, random_state=42)

# Train multiple models and compare
comparison_df = predictor.train_multiple_models()

# Evaluate the best model
predictor.evaluate_model()

# Get feature importance
predictor.get_feature_importance(top_n=20)

# Save the best model
predictor.save_model('best_diagnosis_model.joblib')

PREPARING DATA FOR MODEL TRAINING

[1/3] Encoding target labels...
   ✓ Encoded 15 classes

   Class Mapping:
      0 = allergic_reaction    (59 cases)
      1 = anxiety_attack       (56 cases)
      2 = appendicitis         (69 cases)
      3 = asthma_attack        (83 cases)
      4 = common_cold          (60 cases)
      5 = food_poisoning       (64 cases)
      6 = gastritis            (77 cases)
      7 = influenza            (73 cases)
      8 = kidney_stones        (62 cases)
      9 = migraine             (72 cases)
      10 = pneumonia            (53 cases)
      11 = sinusitis            (61 cases)
      12 = strep_throat         (80 cases)
      13 = tension_headache     (63 cases)
      14 = uti                  (68 cases)

[2/3] Splitting data (train/test = 80/20)...
   ✓ Training set: 800 samples
   ✓ Test set:     200 samples
   ✓ Features:     156

[3/3] Scaling features...
   ✓ Features scaled (mean≈0, std≈1)

DATA PREPARATION COMPLETE ✅
TRAINING 5 MODELS

TRAINING: RA